In [6]:
from finvizfinance.earnings import Earnings
import pandas as pd
from datetime import datetime, timedelta
import yfinance as yf



In [7]:
earnings = Earnings()
earnings._set_period("This Month")

df = earnings.df.copy()

# Sanity check
required_cols = ["Ticker", "Market Cap", "Earnings"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise RuntimeError(f"Missing columns: {missing}")


In [8]:
def parse_market_cap(value):
    if pd.isna(value):
        return 0

    value = str(value).upper().strip()
    multipliers = {
        "T": 1_000_000_000_000,
        "B": 1_000_000_000,
        "M": 1_000_000,
        "K": 1_000,
    }

    suffix = value[-1]
    if suffix in multipliers:
        return float(value[:-1]) * multipliers[suffix]

    try:
        return float(value)
    except ValueError:
        return 0
    
def parse_earnings_date(value):
    if pd.isna(value):
        return None

    # Remove /a or /b suffix
    clean = value.replace("/a", "").replace("/b", "").strip()

    try:
        return datetime.strptime(f"{clean} {datetime.today().year}", "%b %d %Y")
    except ValueError:
        return None

def parse_earnings_timing(value):
    if pd.isna(value):
        return "Unknown"
    value = value.lower()
    if value.endswith("/b"):
        return "Before Market Open"
    if value.endswith("/a"):
        return "After Market Close"
    return "Unknown"


THIS WOULD GENERATE THE EXCEL OUTPUT

In [9]:
df["Market Cap Numeric"] = df["Market Cap"].apply(parse_market_cap)
df["Earnings Date"] = df["Earnings"].apply(parse_earnings_date)
df["Earnings Timing"] = df["Earnings"].apply(parse_earnings_timing)

ONE_BILLION = 1_000_000_000

filtered = (
    df[df["Market Cap Numeric"] >= ONE_BILLION]
    .dropna(subset=["Earnings Date"])
)

# -----------------------
# Top 10 by Market Cap per Day
# -----------------------
final_df = (
    filtered
    .sort_values(["Earnings Date", "Market Cap Numeric"], ascending=[True, False])
    .groupby("Earnings Date")
    .head(10)
    .reset_index(drop=True)
)

# -----------------------
# Select & Format Columns
# -----------------------
final_df = final_df.rename(columns={
    "Ticker": "Ticker",
    "Market Cap": "Market Cap",
    "Earnings Timing": "Earnings Timing"
})

final_df = final_df[
    ["Earnings Date", "Ticker", "Market Cap", "Earnings Timing"]
]

# Format date for Excel
final_df["Earnings Date"] = final_df["Earnings Date"].dt.strftime("%Y-%m-%d")

# -----------------------
# Export to Excel
# -----------------------
month_name = datetime.today().strftime("%B")
output_file = f"earnings_{month_name.lower()}.xlsx"

final_df.to_excel(output_file, index=False)

output_file

'earnings_january.xlsx'

LOAD IN THE EXCEL FILE AND THEN DO ANALYSIS

In [16]:
# -----------------------
# CONFIG
# -----------------------
FILE_PATH = "earnings_january.xlsx"  # or .csv
WEEKS = [3,4]  # 1 | [1,2,3,4] | "all"

# -----------------------
# LOAD FILE
# -----------------------
df = pd.read_csv(FILE_PATH) if FILE_PATH.endswith(".csv") else pd.read_excel(FILE_PATH)
df["Earnings Date"] = pd.to_datetime(df["Earnings Date"])

# -----------------------
# MARKET CAP FORMATTER
# -----------------------
def format_market_cap(n):
    if n >= 1_000_000_000_000:
        return f"{n / 1_000_000_000_000:.1f}T"
    elif n >= 1_000_000_000:
        return f"{n / 1_000_000_000:.1f}B"
    return f"{n:,}"

# -----------------------
# NORMALIZE WEEK INPUT
# -----------------------
if WEEKS == "all":
    weeks = sorted(
        int(w) for w in ((df["Earnings Date"].dt.day - 1) // 7 + 1).unique()
    )
elif isinstance(WEEKS, int):
    weeks = [WEEKS]
else:
    weeks = [int(w) for w in WEEKS]

# -----------------------
# MONTH START
# -----------------------
month_start = df["Earnings Date"].min().replace(day=1)

# -----------------------
# PROCESS EACH WEEK
# -----------------------
for week in weeks:
    week_start = month_start + timedelta(days=(week - 1) * 7)
    week_end = week_start + timedelta(days=6)

    week_df = (
        df[(df["Earnings Date"] >= week_start) & (df["Earnings Date"] <= week_end)]
        .sort_values(["Earnings Date", "Market Cap"], ascending=[True, False])
    )

    if week_df.empty:
        continue

    # -----------------------
    # PRINT OUTPUT
    # -----------------------
    print("=" * 45)
    print(f"Week {week}")
    print(f"Start Date: {week_start.date()}")
    print(f"End Date:   {week_end.date()}\n")

    for date, group in week_df.groupby("Earnings Date"):
        # Assign numeric sort key for timing
        timing_order = {"Before Market Open": 0, "After Market Close": 1, "Unknown": 2}
        group["Timing Sort"] = group["Earnings Timing"].map(timing_order).fillna(2)

        # Sort first by timing, then by Market Cap descending
        group_sorted = group.sort_values(["Timing Sort", "Market Cap"], ascending=[True, False])

        print(date.strftime("%A, %B %d, %Y"))

        for _, row in group_sorted.iterrows():
            print(f"  {row['Ticker']}  —  {format_market_cap(row['Market Cap'])} ({row['Earnings Timing']})")

        print()  # blank line between days


Week 3
Start Date: 2026-01-15
End Date:   2026-01-21

Thursday, January 15, 2026
  TSM  —  1.7T (Before Market Open)
  MS  —  292.8B (Before Market Open)
  GS  —  282.4B (Before Market Open)
  BLK  —  176.4B (Before Market Open)
  FHN  —  11.9B (Before Market Open)
  JBHT  —  19.7B (After Market Close)

Friday, January 16, 2026
  PNC  —  84.2B (Before Market Open)
  STT  —  37.1B (Before Market Open)
  MTB  —  32.5B (Before Market Open)
  WIT  —  30.6B (Before Market Open)
  RF  —  24.7B (Before Market Open)
  BOKF  —  7.8B (After Market Close)

Tuesday, January 20, 2026
  MMM  —  89.8B (Before Market Open)
  USB  —  84.2B (Before Market Open)
  FAST  —  48.1B (Before Market Open)
  DHI  —  47.0B (Before Market Open)
  FITB  —  32.0B (Before Market Open)
  KEY  —  22.8B (Before Market Open)
  NFLX  —  409.0B (After Market Close)
  UAL  —  37.6B (After Market Close)
  IBKR  —  31.5B (After Market Close)
  WTFC  —  9.6B (After Market Close)

Wednesday, January 21, 2026
  JNJ  —  512.8B (

In [39]:
import re

# -----------------------
# CONFIG: Paste your block of text here
# -----------------------
text_block = """
   UNH  —  305.2B (Before Market Open)
  RTX  —  262.7B (Before Market Open)
  BA  —  192.1B (Before Market Open)
  UNP  —  136.4B (Before Market Open)
  HCA  —  108.3B (Before Market Open)
  UPS  —  91.2B (Before Market Open)
  NOC  —  90.8B (Before Market Open)
  GM  —  78.0B (Before Market Open)
  PCAR  —  62.1B (Before Market Open)
"""
# -----------------------
# EXTRACT TICKERS
# -----------------------
# Match any uppercase letters at the start of a line, before the first space or dash
tickers = re.findall(r"^\s*([A-Z]+)\s*—", text_block, re.MULTILINE)

# -----------------------
# OUTPUT
# -----------------------
print("TICKERS=", tickers)


TICKERS= ['UNH', 'RTX', 'BA', 'UNP', 'HCA', 'UPS', 'NOC', 'GM', 'PCAR']


In [40]:
# -----------------------
# CONFIG
# -----------------------
TICKERS= ['UNH', 'RTX', 'BA', 'UNP', 'HCA', 'UPS', 'NOC', 'GM', 'PCAR']

NUM_PREVIOUS_EARNINGS = 6

# -----------------------
# FUNCTION TO CALCULATE EARNINGS REACTIONS
# -----------------------
def earnings_reactions(ticker, num_earnings=4):
    t = yf.Ticker(ticker)
    results = []

    # 1. Get previous earnings dates
    try:
        earnings_df = t.get_earnings_dates(limit=num_earnings)
        if isinstance(earnings_df.index, pd.DatetimeIndex):
            earnings_df = earnings_df.reset_index().rename(columns={"index": "Earnings Date"})
        if "Earnings Date" not in earnings_df.columns:
            if "startdatetime" in earnings_df.columns:
                earnings_df = earnings_df.rename(columns={"startdatetime": "Earnings Date"})
            else:
                earnings_df = earnings_df.rename(columns={earnings_df.columns[0]: "Earnings Date"})
        earnings_df = earnings_df.head(num_earnings)
    except Exception as e:
        print(f"Could not fetch earnings dates for {ticker}: {e}")
        return []

    # 2. For each earnings date, fetch OHLC for day before, day of, day after
    for _, row in earnings_df.iterrows():
        edate = pd.to_datetime(row["Earnings Date"]).date()
        start = edate - pd.Timedelta(days=1)
        end = edate + pd.Timedelta(days=1)

        hist = t.history(start=start, end=end + pd.Timedelta(days=1), auto_adjust=False)
        if hist.empty:
            continue

        # Ensure dates are indexed properly
        hist.index = hist.index.date

        day_before = hist.loc[hist.index == edate - pd.Timedelta(days=1)]
        day_of = hist.loc[hist.index == edate]
        day_after = hist.loc[hist.index == edate + pd.Timedelta(days=1)]

        close_before = day_before["Close"].iloc[0] if not day_before.empty else None
        close_day = day_of["Close"].iloc[0] if not day_of.empty else None
        close_after = day_after["Close"].iloc[0] if not day_after.empty else None

        pct_before_day = (close_day - close_before)/close_before*100 if close_before and close_day else None
        pct_day_after = (close_after - close_day)/close_day*100 if close_after and close_day else None
        pct_before_after = (close_after - close_before)/close_before*100 if close_after and close_before else None

        results.append({
            "Ticker": ticker,
            "Earnings Date": edate,
            "Close Before": close_before,
            "Close DayOf": close_day,
            "Close After": close_after,
            "% Before→DayOf": round(pct_before_day, 2) if pct_before_day is not None else None,
            "% DayOf→After": round(pct_day_after, 2) if pct_day_after is not None else None,
            "% Before→After": round(pct_before_after, 2) if pct_before_after is not None else None,
        })

    return results

# -----------------------
# RUN FUNCTION FOR MULTIPLE TICKERS
# -----------------------
all_results = []

for ticker in TICKERS:
    print(f"Fetching earnings for {ticker}...")
    all_results.extend(earnings_reactions(ticker, NUM_PREVIOUS_EARNINGS))

# -----------------------
# CREATE DATAFRAME
# -----------------------
if all_results:
    df_results = pd.DataFrame(all_results)
    df_results["Earnings Date"] = pd.to_datetime(df_results["Earnings Date"])
    df_results = df_results.sort_values(["Ticker", "Earnings Date"], ascending=[True, False])
    df_results.reset_index(drop=True, inplace=True)
    display(df_results)
else:
    print("No historical earnings data found.")

Fetching earnings for UNH...
Fetching earnings for RTX...
Fetching earnings for BA...
Fetching earnings for UNP...
Fetching earnings for HCA...
Fetching earnings for UPS...
Fetching earnings for NOC...
Fetching earnings for GM...
Fetching earnings for PCAR...


,Ticker,Earnings Date,Close Before,Close DayOf,Close After,% Before→DayOf,% DayOf→After,% Before→After
0,BA,2026-01-27,250.279999,NaN,NaN,NaN,NaN,NaN
1,BA,2025-10-29,223.330002,213.580002,200.080002,-4.37,-6.32,-10.41
2,BA,2025-07-29,236.410004,226.080002,225.839996,-4.37,-0.11,-4.47
3,BA,2025-04-23,162.520004,172.369995,176.259995,6.06,2.26,8.45
4,BA,2025-01-28,175.160004,177.779999,173.660004,1.50,-2.32,-0.86
5,BA,2024-10-23,159.880005,157.059998,155.199997,-1.76,-1.18,-2.93
6,GM,2026-01-27,79.529999,NaN,NaN,NaN,NaN,NaN
7,GM,2025-10-21,58.000000,66.620003,67.309998,14.86,1.04,16.05
8,GM,2025-07-22,53.209999,48.889999,53.130001,-8.12,8.67,-0.15
9,GM,2025-04-29,47.240002,46.939999,45.240002,-0.64,-3.62,-4.23


In [41]:
# -----------------------
# GAMBLE ANALYSIS
# -----------------------
import numpy as np

# Only consider rows with valid Before→After data
df_valid = df_results.dropna(subset=["% Before→After"])

# Group by ticker
summary = df_valid.groupby("Ticker").agg(
    avg_before_after = ("% Before→After", "mean"),
    median_before_after = ("% Before→After", "median"),
    avg_before_day = ("% Before→DayOf", "mean"),
    avg_abs_move = ("% Before→After", lambda x: np.mean(np.abs(x))),
    num_earnings = ("Earnings Date", "count"),
    pos_moves = ("% Before→After", lambda x: (x > 0).sum()),
    neg_moves = ("% Before→After", lambda x: (x < 0).sum())
)

# Percent positive and negative
summary["pct_pos"] = summary["pos_moves"] / summary["num_earnings"] * 100
summary["pct_neg"] = summary["neg_moves"] / summary["num_earnings"] * 100

# Simple gamble flag
# Criteria: Historically positive > 60% of earnings, and average swing > 5%
summary["gamble_flag"] = summary.apply(
    lambda row: "Yes" if row["pct_pos"] >= 60 and row["avg_abs_move"] >= 5 else "No",
    axis=1
)

# Sort by avg_before_after to see best historically performing tickers
summary = summary.sort_values(["gamble_flag", "avg_before_after"], ascending=[False, False])

# Display
display(summary[[
    "num_earnings", "avg_before_after", "median_before_after",
    "avg_before_day", "avg_abs_move", "pct_pos", "pct_neg", "gamble_flag"
]])


,num_earnings,avg_before_after,median_before_after,avg_before_day,avg_abs_move,pct_pos,pct_neg,gamble_flag
Ticker,,,,,,,,
RTX,5,2.0940,1.040,-0.274,3.9460,80.0,20.0,No
GM,5,2.0920,-0.150,1.404,7.5880,40.0,60.0,No
PCAR,5,0.9620,0.890,-0.038,3.8780,60.0,40.0,No
NOC,5,0.0260,0.570,-0.380,4.8100,60.0,40.0,No
UNP,5,-1.7140,-2.950,-1.602,3.9140,20.0,80.0,No
BA,5,-2.0440,-2.930,-0.588,5.4240,20.0,80.0,No
UPS,5,-3.3760,-1.840,-2.354,8.8760,40.0,60.0,No
UNH,4,-5.1125,-5.665,-5.275,5.1125,0.0,100.0,No


In [92]:
# -----------------------
# CONFIG
# -----------------------
TICKERS = ['STZ', 'APLD']  # <-- enter any tickers you want
NUM_PREVIOUS_EARNINGS = 8

# -----------------------
# FUNCTION TO CALCULATE EARNINGS REACTIONS
# -----------------------
def earnings_reactions(ticker, num_earnings=4):
    t = yf.Ticker(ticker)
    results = []

    # 1. Get previous earnings dates
    try:
        earnings_df = t.get_earnings_dates(limit=num_earnings)
        if isinstance(earnings_df.index, pd.DatetimeIndex):
            earnings_df = earnings_df.reset_index().rename(columns={"index": "Earnings Date"})
        if "Earnings Date" not in earnings_df.columns:
            if "startdatetime" in earnings_df.columns:
                earnings_df = earnings_df.rename(columns={"startdatetime": "Earnings Date"})
            else:
                earnings_df = earnings_df.rename(columns={earnings_df.columns[0]: "Earnings Date"})
        earnings_df = earnings_df.head(num_earnings)
    except Exception as e:
        print(f"Could not fetch earnings dates for {ticker}: {e}")
        return []

    # 2. For each earnings date, fetch OHLC for day before, day of, day after
    for _, row in earnings_df.iterrows():
        edate = pd.to_datetime(row["Earnings Date"]).date()
        start = edate - pd.Timedelta(days=1)
        end = edate + pd.Timedelta(days=1)

        hist = t.history(start=start, end=end + pd.Timedelta(days=1), auto_adjust=False)
        if hist.empty:
            continue

        # Ensure dates are indexed properly
        hist.index = hist.index.date

        day_before = hist.loc[hist.index == edate - pd.Timedelta(days=1)]
        day_of = hist.loc[hist.index == edate]
        day_after = hist.loc[hist.index == edate + pd.Timedelta(days=1)]

        close_before = day_before["Close"].iloc[0] if not day_before.empty else None
        close_day = day_of["Close"].iloc[0] if not day_of.empty else None
        close_after = day_after["Close"].iloc[0] if not day_after.empty else None

        pct_before_day = (close_day - close_before)/close_before*100 if close_before and close_day else None
        pct_day_after = (close_after - close_day)/close_day*100 if close_after and close_day else None
        pct_before_after = (close_after - close_before)/close_before*100 if close_after and close_before else None

        results.append({
            "Ticker": ticker,
            "Earnings Date": edate,
            "Close Before": close_before,
            "Close DayOf": close_day,
            "Close After": close_after,
            "% Before→DayOf": round(pct_before_day, 2) if pct_before_day is not None else None,
            "% DayOf→After": round(pct_day_after, 2) if pct_day_after is not None else None,
            "% Before→After": round(pct_before_after, 2) if pct_before_after is not None else None,
        })

    return results

# -----------------------
# RUN FUNCTION FOR MULTIPLE TICKERS
# -----------------------
all_results = []

for ticker in TICKERS:
    print(f"Fetching earnings for {ticker}...")
    all_results.extend(earnings_reactions(ticker, NUM_PREVIOUS_EARNINGS))

# -----------------------
# CREATE DATAFRAME
# -----------------------
if all_results:
    df_results = pd.DataFrame(all_results)
    df_results["Earnings Date"] = pd.to_datetime(df_results["Earnings Date"])
    df_results = df_results.sort_values(["Ticker", "Earnings Date"], ascending=[True, False])
    df_results.reset_index(drop=True, inplace=True)
    display(df_results)
else:
    print("No historical earnings data found.")


# -----------------------
# GAMBLE ANALYSIS
# -----------------------
import numpy as np

# Only consider rows with valid Before→After data
df_valid = df_results.dropna(subset=["% Before→After"])

# Group by ticker
summary = df_valid.groupby("Ticker").agg(
    avg_before_after = ("% Before→After", "mean"),
    median_before_after = ("% Before→After", "median"),
    avg_before_day = ("% Before→DayOf", "mean"),
    avg_abs_move = ("% Before→After", lambda x: np.mean(np.abs(x))),
    num_earnings = ("Earnings Date", "count"),
    pos_moves = ("% Before→After", lambda x: (x > 0).sum()),
    neg_moves = ("% Before→After", lambda x: (x < 0).sum())
)

# Percent positive and negative
summary["pct_pos"] = summary["pos_moves"] / summary["num_earnings"] * 100
summary["pct_neg"] = summary["neg_moves"] / summary["num_earnings"] * 100

# Simple gamble flag
# Criteria: Historically positive > 60% of earnings, and average swing > 5%
summary["gamble_flag"] = summary.apply(
    lambda row: "Yes" if row["pct_pos"] >= 60 and row["avg_abs_move"] >= 5 else "No",
    axis=1
)

# Sort by avg_before_after to see best historically performing tickers
summary = summary.sort_values(["gamble_flag", "avg_before_after"], ascending=[False, False])

# Display
display(summary[[
    "num_earnings", "avg_before_after", "median_before_after",
    "avg_before_day", "avg_abs_move", "pct_pos", "pct_neg", "gamble_flag"
]])


Fetching earnings for STZ...


$STZ: possibly delisted; no price data found  (1d 2026-04-08 -> 2026-04-11) (Yahoo error = "Data doesn't exist for startDate = 1775620800, endDate = 1775880000")


Fetching earnings for APLD...


$APLD: possibly delisted; no price data found  (1d 2026-04-08 -> 2026-04-11) (Yahoo error = "Data doesn't exist for startDate = 1775620800, endDate = 1775880000")


,Ticker,Earnings Date,Close Before,Close DayOf,Close After,% Before→DayOf,% DayOf→After,% Before→After
0,APLD,2026-01-07,30.260000,29.559999,31.940001,-2.31,8.05,5.55
1,APLD,2025-10-09,27.940001,29.290001,33.990002,4.83,16.05,21.65
2,APLD,2025-07-30,10.120000,10.030000,13.140000,-0.89,31.01,29.84
3,APLD,2025-04-14,NaN,5.370000,3.440000,NaN,-35.94,NaN
4,APLD,2025-01-14,7.765000,8.540000,8.370000,9.98,-1.99,7.79
5,APLD,2024-10-09,7.850000,7.400000,6.890000,-5.73,-6.89,-12.23
6,APLD,2024-08-28,4.730000,4.410000,3.820000,-6.77,-13.38,-19.24
7,STZ,2026-01-07,143.649994,140.490005,147.960007,-2.20,5.32,3.00
8,STZ,2025-10-06,NaN,138.710007,140.139999,NaN,1.03,NaN
9,STZ,2025-07-01,162.679993,166.419998,173.869995,2.30,4.48,6.88


,num_earnings,avg_before_after,median_before_after,avg_before_day,avg_abs_move,pct_pos,pct_neg,gamble_flag
Ticker,,,,,,,,
APLD,6,5.5600,6.67,-0.148333,16.0500,66.666667,33.333333,Yes
STZ,4,3.6075,4.94,0.670000,5.3675,75.000000,25.000000,Yes
